In [2]:
import pandas as pd
import numpy as np
from collections import Counter
import os
import pandas as pd
import csv
from collections import defaultdict

folder = r"/Users/vaidehipandya/Desktop/Work/loaded_data"  # <-- adjust if needed
os.chdir(folder)

files = ["pa_1.csv", "pa_2.csv", "pa_3.csv"]

In [2]:


files = ["pa_1.csv", "pa_2.csv", "pa_3.csv"]

for f in files:
    df = pd.read_csv(f)
    
    # Normalize column names (remove spaces, tabs, invisible characters)
    df.columns = df.columns.str.strip().str.replace("\t", "", regex=False).str.replace("\xa0", "", regex=False)
    
    # Re-check which columns look unnamed
    drop_cols = [c for c in df.columns if "unnamed" in c.lower()]
    
    if drop_cols:
        print(f"{f}: Dropping {drop_cols}")
        df.drop(columns=drop_cols, inplace=True, errors="ignore")
    else:
        print(f"{f}: No unnamed columns found")

    out = f.replace(".csv", "_clean.csv")
    df.to_csv(out, index=False, encoding="utf-8")
    print(f"Saved cleaned file {out}\n")

pa_1.csv: No unnamed columns found
Saved cleaned file pa_1_clean.csv

pa_2.csv: No unnamed columns found
Saved cleaned file pa_2_clean.csv

pa_3.csv: Dropping ['Unnamed: 15', 'Unnamed: 16']
Saved cleaned file pa_3_clean.csv



In [3]:
column_map = {
    "Malignant_Epithelium": "malignant_label",
    "Cluster": "hpc_id",
    "Predominant_Pattern": "malignant_subtype",
    "Predominant_Pattern ": "malignant_subtype",
    "Second_Pattern_(if present)": "second_malignant_subtype",
    "Second_Pattern _(if present)": "second_malignant_subtype",
    "Second_Patter_(if present)": "second_malignant_subtype",
    "Inflammation*": "inflammation_label",
    "Inflammation*.1": "inflammation_label",
    "Necrosis**": "necrosis_label",
    "Necrosis**.1": "necrosis_label",
    "Stroma_Epithelium_Ratio": "stromal_epithelium_ratio",
    "Stroma_ Epithelium_Ratio": "stromal_epithelium_ratio",
    "Stromal_cellularity***": "stromal_cellularity",
    "Tiles_contain_mostly": "non_malignant_subtype",
    "Second_most_common_feature": "second_non_malignant_subtype",
    "Second_most_common feature": "second_non_malignant_subtype",
    "Second_most_common_feature ": "second_non_malignant_subtype",
    "Percentage cluster purity": "cluster_homogeneity",
    "Percentage_cluster_purity": "cluster_homogeneity",
}

In [4]:
dfs = []
for f in ["pa_1_clean.csv", "pa_2_clean.csv", "pa_3_clean.csv"]:
    df = pd.read_csv(f, encoding="utf-8")          # files already clean
    df.rename(columns=column_map, inplace=True)    # unify names
    df.columns = df.columns.str.strip()            # remove stray spaces

    # combine duplicate inflammation/necrosis columns if any
    infl_cols = [c for c in df.columns if "inflammation" in c.lower()]
    if len(infl_cols) > 1:
        df["inflammation_label"] = df[infl_cols].bfill(axis=1).iloc[:, 0]
        df.drop(columns=[c for c in infl_cols if c != "inflammation_label"], inplace=True)

    necr_cols = [c for c in df.columns if "necrosis" in c.lower()]
    if len(necr_cols) > 1:
        df["necrosis_label"] = df[necr_cols].bfill(axis=1).iloc[:, 0]
        df.drop(columns=[c for c in necr_cols if c != "necrosis_label"], inplace=True)

    df["hpc_id"] = pd.to_numeric(df["hpc_id"], errors="coerce").fillna(-1).astype(int)
    df.reset_index(drop=True, inplace=True)

    print(f" Cleaned {f} — Columns:", list(df.columns))
    dfs.append(df)

p1, p2, p3 = dfs

 Cleaned pa_1_clean.csv — Columns: ['malignant_label', 'hpc_id', 'malignant_subtype', 'second_malignant_subtype', 'inflammation_label', 'necrosis_label', 'stromal_epithelium_ratio', 'stromal_cellularity', 'Other comments', 'non_malignant_subtype', 'second_non_malignant_subtype', 'inflammation_label', 'necrosis_label', 'Other_notable_features_or_comments?', 'cluster_homogeneity']
 Cleaned pa_2_clean.csv — Columns: ['malignant_label', 'hpc_id', 'malignant_subtype', 'second_malignant_subtype', 'inflammation_label', 'necrosis_label', 'stromal_epithelium_ratio', 'stromal_cellularity', 'Other comments', 'non_malignant_subtype', 'second_non_malignant_subtype', 'inflammation_label', 'necrosis_label', 'Other_notable_features_or_comments?', 'cluster_homogeneity']
 Cleaned pa_3_clean.csv — Columns: ['malignant_label', 'hpc_id', 'malignant_subtype', 'second_malignant_subtype', 'inflammation_label', 'necrosis_label', 'stromal_epithelium_ratio', 'stromal_cellularity', 'Other_comments', 'non_malignan

/var/folders/q4/sypcwz3j32b0qyjm3q7rk6cc0000gn/T/ipykernel_83768/533215669.py:15: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["necrosis_label"] = df[necr_cols].bfill(axis=1).iloc[:, 0]
/var/folders/q4/sypcwz3j32b0qyjm3q7rk6cc0000gn/T/ipykernel_83768/533215669.py:15: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["necrosis_label"] = df[necr_cols].bfill(axis=1).iloc[:, 0]
/var/folders/q4/sypcwz3j32b0qyjm3q7rk6cc0000gn/T/ipykernel_83768/533215669.py:15: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will c

In [5]:

files = ["pa_1_clean.csv", "pa_2_clean.csv", "pa_3_clean.csv"]

for f in files:
    df = pd.read_csv(f)

    # Drop any variant of comment columns
    drop_cols = [c for c in df.columns if "other" in c.lower() and "comment" in c.lower()]
    if drop_cols:
        print(f"{f}: Dropping {drop_cols}")
        df.drop(columns=drop_cols, inplace=True, errors="ignore")
    else:
        print(f"{f}: No comment columns found")

    # Save the final version
    out = f.replace("_clean.csv", "_final.csv")
    df.to_csv(out, index=False, encoding="utf-8")
    print(f"Saved → {out} with {len(df.columns)} columns\n")

pa_1_clean.csv: Dropping ['Other comments', 'Other_notable_features_or_comments?']
Saved → pa_1_final.csv with 13 columns

pa_2_clean.csv: Dropping ['Other comments', 'Other_notable_features_or_comments?']
Saved → pa_2_final.csv with 13 columns

pa_3_clean.csv: Dropping ['Other_comments', 'Other_notable_features_or_comments?']
Saved → pa_3_final.csv with 13 columns



In [6]:
for f in ["pa_1_final.csv", "pa_2_final.csv", "pa_3_final.csv"]:
    df = pd.read_csv(f)
    print(f"\n=== {f} ===")
    print(df.columns.tolist())


=== pa_1_final.csv ===
['Malignant_Epithelium', 'Cluster', 'Predominant_Pattern', 'Second_Pattern _(if present)', 'Inflammation*', 'Necrosis**', 'Stroma_ Epithelium_Ratio', 'Stromal_cellularity***', 'Tiles_contain_mostly', 'Second_most_common_feature', 'Inflammation*.1', 'Necrosis**.1', 'Percentage cluster purity']

=== pa_2_final.csv ===
['Malignant_Epithelium', 'Cluster', 'Predominant_Pattern', 'Second_Pattern_(if present)', 'Inflammation*', 'Necrosis**', 'Stroma_Epithelium_Ratio', 'Stromal_cellularity***', 'Tiles_contain_mostly', 'Second_most_common_feature', 'Inflammation*.1', 'Necrosis**.1', 'Percentage cluster purity']

=== pa_3_final.csv ===
['Malignant_Epithelium', 'Cluster', 'Predominant_Pattern', 'Second_Patter_(if present)', 'Inflammation*', 'Necrosis**', 'Stroma_Epithelium_Ratio', 'Stromal_cellularity***', 'Tiles_contain_mostly', 'Second_most_common_feature', 'Inflammation*.1', 'Necrosis**.1', 'Percentage_cluster_purity']


In [110]:
import pandas as pd

for f in ["pa_1_merged.csv", "pa_2_merged.csv", "pa_3_merged.csv"]:
    df = pd.read_csv(f, nrows=2)
    print("\nFile:", f)
    print("Columns:", list(df.columns))


File: pa_1_merged.csv
Columns: ['malignant_label', 'hpc_id', 'malignant_subtype', 'second_malignant_subtype', 'inflammation_label', 'necrosis_label', 'stromal_epithelium_ratio', 'stromal_cellularity', 'non_malignant_subtype', 'second_non_malignant_subtype', 'cluster_homogeneity']

File: pa_2_merged.csv
Columns: ['malignant_label', 'hpc_id', 'malignant_subtype', 'second_malignant_subtype', 'inflammation_label', 'necrosis_label', 'stromal_epithelium_ratio', 'stromal_cellularity', 'non_malignant_subtype', 'second_non_malignant_subtype', 'cluster_homogeneity']

File: pa_3_merged.csv
Columns: ['malignant_label', 'hpc_id', 'malignant_subtype', 'second_malignant_subtype', 'inflammation_label', 'necrosis_label', 'stromal_epithelium_ratio', 'stromal_cellularity', 'non_malignant_subtype', 'second_non_malignant_subtype', 'cluster_homogeneity']


In [8]:
import pandas as pd

label_exclude = ["samples", "slides", "tiles", "sample_id", "slide_id", 
                 "tile_name", "hpc_id", "leiden_2.5", "leiden_2_5"]

for f in ["pa_1_merged.csv", "pa_2_merged.csv", "pa_3_merged.csv"]:
    df = pd.read_csv(f)

    print(f"\n=== {f} ===")

    # 1. Malignant-related columns
    malignant_cols = [c for c in df.columns if "malignant" in c.lower()]
    print("Malignant-related columns:", malignant_cols)

    # 2. All label columns (anything that is not metadata)
    label_cols = [c for c in df.columns if c.lower() not in label_exclude]
    print("All label columns:", label_cols)


=== pa_1_merged.csv ===
Malignant-related columns: ['malignant_label', 'malignant_subtype', 'second_malignant_subtype', 'non_malignant_subtype', 'second_non_malignant_subtype']
All label columns: ['malignant_label', 'malignant_subtype', 'second_malignant_subtype', 'inflammation_label', 'necrosis_label', 'stromal_epithelium_ratio', 'stromal_cellularity', 'non_malignant_subtype', 'second_non_malignant_subtype', 'cluster_homogeneity']

=== pa_2_merged.csv ===
Malignant-related columns: ['malignant_label', 'malignant_subtype', 'second_malignant_subtype', 'non_malignant_subtype', 'second_non_malignant_subtype']
All label columns: ['malignant_label', 'malignant_subtype', 'second_malignant_subtype', 'inflammation_label', 'necrosis_label', 'stromal_epithelium_ratio', 'stromal_cellularity', 'non_malignant_subtype', 'second_non_malignant_subtype', 'cluster_homogeneity']

=== pa_3_merged.csv ===
Malignant-related columns: ['malignant_label', 'malignant_subtype', 'second_malignant_subtype', 'non_

In [ ]:
def safe_merge(file1, file2, file3, key="hpc_id"):
    store = defaultdict(dict)

    # read each file and map by hpc_id
    for i, f in enumerate([file1, file2, file3], start=1):
        print(f"Reading {f} ...")
        with open(f, newline="", encoding="utf-8") as fh:
            r = csv.DictReader(fh)
            for row in r:
                try:
                    k = int(float(row[key]))  # robust numeric conversion
                except Exception:
                    continue
                for c, v in row.items():
                    if c != key:
                        store[k][f"{c}_p{i}"] = v

    out = "merged_manual.csv"
    fieldnames = [key] + sorted({c for d in store.values() for c in d})
    with open(out, "w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=fieldnames)
        w.writeheader()
        for k, cols in store.items():
            row = {key: k, **cols}
            w.writerow(row)

    print(f" Wrote merged file: {out}")

safe_merge("pa_1_merged.csv", "pa_2_merged.csv", "pa_3_merged.csv")

Reading pa_1_merged.csv ...
Reading pa_2_merged.csv ...
Reading pa_3_merged.csv ...
✅ Wrote merged file: merged_manual.csv


In [10]:
merged = pd.read_csv("merged_manual.csv")
for key in ["malignant_subtype", "necrosis_label", "inflammation_label"]:
    cols = [c for c in merged.columns if key in c]
    print(f"{key} columns:", cols)

malignant_subtype columns: ['malignant_subtype_p1', 'malignant_subtype_p2', 'malignant_subtype_p3', 'non_malignant_subtype_p1', 'non_malignant_subtype_p2', 'non_malignant_subtype_p3', 'second_malignant_subtype_p1', 'second_malignant_subtype_p2', 'second_malignant_subtype_p3', 'second_non_malignant_subtype_p1', 'second_non_malignant_subtype_p2', 'second_non_malignant_subtype_p3']
necrosis_label columns: ['necrosis_label_p1', 'necrosis_label_p2', 'necrosis_label_p3']
inflammation_label columns: ['inflammation_label_p1', 'inflammation_label_p2', 'inflammation_label_p3']


In [11]:
print("Number of columns:", len(merged.columns))
print("All column names:")
print(list(merged.columns))

Number of columns: 31
All column names:
['hpc_id', 'cluster_homogeneity_p1', 'cluster_homogeneity_p2', 'cluster_homogeneity_p3', 'inflammation_label_p1', 'inflammation_label_p2', 'inflammation_label_p3', 'malignant_label_p1', 'malignant_label_p2', 'malignant_label_p3', 'malignant_subtype_p1', 'malignant_subtype_p2', 'malignant_subtype_p3', 'necrosis_label_p1', 'necrosis_label_p2', 'necrosis_label_p3', 'non_malignant_subtype_p1', 'non_malignant_subtype_p2', 'non_malignant_subtype_p3', 'second_malignant_subtype_p1', 'second_malignant_subtype_p2', 'second_malignant_subtype_p3', 'second_non_malignant_subtype_p1', 'second_non_malignant_subtype_p2', 'second_non_malignant_subtype_p3', 'stromal_cellularity_p1', 'stromal_cellularity_p2', 'stromal_cellularity_p3', 'stromal_epithelium_ratio_p1', 'stromal_epithelium_ratio_p2', 'stromal_epithelium_ratio_p3']


In [12]:


# Load merged file
merged = pd.read_csv("merged_manual.csv")

# --- Helper function: majority vote ---
def majority_vote(values):
    values = [v for v in values if pd.notna(v)]
    if not values:
        return None
    return Counter(values).most_common(1)[0][0]

# --- Helper function: agreement score ---
def agreement_score(values):
    values = [v for v in values if pd.notna(v)]
    if len(values) <= 1:
        return 1.0 if values else 0.0
    most_common = Counter(values).most_common(1)[0][1]
    return round(most_common / len(values), 2)

# --- Core fields to process ---
base_labels = [
    "malignant_label",
    "malignant_subtype",
    "second_malignant_subtype",
    "non_malignant_subtype",
    "second_non_malignant_subtype",
    "inflammation_label",
    "necrosis_label",
    "stromal_epithelium_ratio",
    "stromal_cellularity",
    "cluster_homogeneity"
]

# --- Compute majority & agreement per field ---
results = merged[["hpc_id"]].copy()

for label in base_labels:
    cols = [c for c in merged.columns if c.startswith(label + "_p")]
    if len(cols) == 0:
        print(f"⚠️ Skipping {label} (no columns found)")
        continue

    results[label + "_majority"] = merged[cols].apply(majority_vote, axis=1)
    results[label + "_agreement"] = merged[cols].apply(agreement_score, axis=1)
    print(f" Processed {label}: {len(cols)} annotators")

# --- Save the consensus table ---
results.to_csv("merged_majority_vote.csv", index=False)
print("\n Saved majority-vote consensus to merged_majority_vote.csv")

 Processed malignant_label: 3 annotators
 Processed malignant_subtype: 3 annotators
 Processed second_malignant_subtype: 3 annotators
 Processed non_malignant_subtype: 3 annotators
 Processed second_non_malignant_subtype: 3 annotators
 Processed inflammation_label: 3 annotators
 Processed necrosis_label: 3 annotators
 Processed stromal_epithelium_ratio: 3 annotators
 Processed stromal_cellularity: 3 annotators
 Processed cluster_homogeneity: 3 annotators

 Saved majority-vote consensus to merged_majority_vote.csv


In [13]:


# Read the consensus file
results = pd.read_csv("merged_majority_vote.csv")

print(" File loaded successfully.")
print("Shape:", results.shape)
display(results.head(100))

 File loaded successfully.
Shape: (72, 21)


,hpc_id,malignant_label_majority,malignant_label_agreement,malignant_subtype_majority,malignant_subtype_agreement,second_malignant_subtype_majority,second_malignant_subtype_agreement,non_malignant_subtype_majority,non_malignant_subtype_agreement,second_non_malignant_subtype_majority,...,inflammation_label_majority,inflammation_label_agreement,necrosis_label_majority,necrosis_label_agreement,stromal_epithelium_ratio_majority,stromal_epithelium_ratio_agreement,stromal_cellularity_majority,stromal_cellularity_agreement,cluster_homogeneity_majority,cluster_homogeneity_agreement
0,0,Yes,1.0,Acinar,0.67,Lepidic,0.5,NaN,0.00,NaN,...,Mild-moderate,0.67,None,1.0,Roughly equal,1.00,Moderate,0.67,High (> 75%),1.00
1,1,No,1.0,NaN,0.00,NaN,0.0,Collagenosis,0.67,Vessels (wall or lumen),...,None-sparse,1.00,None,1.0,NaN,0.00,NaN,0.00,High (> 75%),1.00
2,2,No,1.0,NaN,0.00,NaN,0.0,Other (please specify),0.33,Vessels (wall or lumen),...,None-sparse,0.33,None,1.0,NaN,0.00,NaN,0.00,High (> 75%),1.00
3,3,No,1.0,NaN,0.00,NaN,0.0,Collagenosis,1.00,Vessels (wall or lumen),...,None-sparse,1.00,None,1.0,NaN,0.00,NaN,0.00,High (> 75%),1.00
4,4,No,1.0,NaN,0.00,NaN,0.0,Cartilage,1.00,NaN,...,None-sparse,1.00,None,1.0,NaN,0.00,NaN,0.00,High (> 75%),1.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,67,Yes,1.0,Acinar,1.00,Lepidic,1.0,NaN,0.00,NaN,...,Marked,1.00,None,1.0,Roughly equal,0.67,Moderate,0.67,High (> 75%),0.67
68,68,Yes,1.0,Acinar,1.00,Micropapillary,0.5,NaN,0.00,NaN,...,Mild-moderate,0.67,None,1.0,Roughly equal,0.33,Low,0.33,High (> 75%),1.00
69,69,Yes,1.0,Acinar,1.00,Cribriform,1.0,NaN,0.00,NaN,...,Mild-moderate,1.00,None,1.0,More stroma,0.67,Moderate,0.67,High (> 75%),0.67
70,70,No,1.0,NaN,0.00,NaN,0.0,Non-neoplastic pathological lung (please specify),1.00,NaN,...,Marked,0.50,None,1.0,NaN,0.00,NaN,0.00,Moderate (50 - 75%),0.67


In [14]:
import pandas as pd

# Load the majority-vote results
consensus = pd.read_csv("merged_majority_vote.csv")

# Optional: keep only HPCs where pathologists agreed (≥ 0.67)
#consensus = consensus[consensus["malignant_label_agreement"] >= 0.67]

# Drop all _agreement columns (they were only metadata)
consensus = consensus.drop(columns=[c for c in consensus.columns if c.endswith("_agreement")])

# Save the cleaned consensus table
consensus.to_csv("HPC_consensus_labels.csv", index=False)
print("Saved clean HPC-level labels to HPC_consensus_labels.csv")

Saved clean HPC-level labels to HPC_consensus_labels.csv


In [ ]:
import pandas as pd

# Load your consensus file (the one after dropping _agreement)
consensus = pd.read_csv("HPC_consensus_labels.csv")

# Rename columns by stripping "_majority"
consensus.rename(columns=lambda x: x.replace("_majority", ""), inplace=True)

# Save the updated version
consensus.to_csv("HPC_consensus_labels_clean.csv", index=False, encoding="utf-8")

print("Cleaned column names — saved as HPC_consensus_labels_clean.csv")
print("Columns now:", list(consensus.columns))

✅ Cleaned column names — saved as HPC_consensus_labels_clean.csv
Columns now: ['hpc_id', 'malignant_label', 'malignant_subtype', 'second_malignant_subtype', 'non_malignant_subtype', 'second_non_malignant_subtype', 'inflammation_label', 'necrosis_label', 'stromal_epithelium_ratio', 'stromal_cellularity', 'cluster_homogeneity']


In [16]:
results = pd.read_csv("HPC_consensus_labels_clean.csv")

print(" File loaded successfully.")
print("Shape:", results.shape)
display(results.head(71))

 File loaded successfully.
Shape: (72, 11)


,hpc_id,malignant_label,malignant_subtype,second_malignant_subtype,non_malignant_subtype,second_non_malignant_subtype,inflammation_label,necrosis_label,stromal_epithelium_ratio,stromal_cellularity,cluster_homogeneity
0,0,Yes,Acinar,Lepidic,NaN,NaN,Mild-moderate,None,Roughly equal,Moderate,High (> 75%)
1,1,No,NaN,NaN,Collagenosis,Vessels (wall or lumen),None-sparse,None,NaN,NaN,High (> 75%)
2,2,No,NaN,NaN,Other (please specify),Vessels (wall or lumen),None-sparse,None,NaN,NaN,High (> 75%)
3,3,No,NaN,NaN,Collagenosis,Vessels (wall or lumen),None-sparse,None,NaN,NaN,High (> 75%)
4,4,No,NaN,NaN,Cartilage,NaN,None-sparse,None,NaN,NaN,High (> 75%)
...,...,...,...,...,...,...,...,...,...,...,...
66,66,No,NaN,NaN,Normal/near-normal lung,NaN,None-sparse,None,NaN,NaN,High (> 75%)
67,67,Yes,Acinar,Lepidic,NaN,NaN,Marked,None,Roughly equal,Moderate,High (> 75%)
68,68,Yes,Acinar,Micropapillary,NaN,NaN,Mild-moderate,None,Roughly equal,Low,High (> 75%)
69,69,Yes,Acinar,Cribriform,NaN,NaN,Mild-moderate,None,More stroma,Moderate,High (> 75%)


In [17]:



import pandas as pd

hpc_df = pd.read_csv("HPC_consensus_labels_clean.csv")

# show the tail to confirm
print(hpc_df.tail(3))

# drop last row
hpc_df = hpc_df.drop(hpc_df.tail(1).index).reset_index(drop=True)

print("New shape:", hpc_df.shape)
print(hpc_df.tail(3))

# save
hpc_df.to_csv("HPC_consensus_labels_clean.csv", index=False)

    hpc_id malignant_label malignant_subtype second_malignant_subtype  \
69      69             Yes            Acinar               Cribriform   
70      70              No               NaN                      NaN   
71      -1             NaN               NaN                      NaN   

                                non_malignant_subtype  \
69                                                NaN   
70  Non-neoplastic pathological lung (please specify)   
71                                                NaN   

   second_non_malignant_subtype inflammation_label necrosis_label  \
69                          NaN      Mild-moderate         None     
70                          NaN             Marked         None     
71                          NaN                NaN            NaN   

   stromal_epithelium_ratio stromal_cellularity  cluster_homogeneity  
69              More stroma            Moderate         High (> 75%)  
70                      NaN                 NaN  Moderate (

In [18]:
import os
os.getcwd()

'/Users/vaidehipandya/Desktop/Work/loaded_data'

In [158]:
display(results.head (70))

,hpc_id,malignant_label,malignant_subtype,second_malignant_subtype,non_malignant_subtype,second_non_malignant_subtype,inflammation_label,necrosis_label,stromal_epithelium_ratio,stromal_cellularity,cluster_homogeneity
0,0,Yes,Acinar,Lepidic,NaN,NaN,Mild-moderate,None,Roughly equal,Moderate,High (> 75%)
1,1,No,NaN,NaN,Collagenosis,Vessels (wall or lumen),None-sparse,None,NaN,NaN,High (> 75%)
2,2,No,NaN,NaN,Other (please specify),Vessels (wall or lumen),None-sparse,None,NaN,NaN,High (> 75%)
3,3,No,NaN,NaN,Collagenosis,Vessels (wall or lumen),None-sparse,None,NaN,NaN,High (> 75%)
4,4,No,NaN,NaN,Cartilage,NaN,None-sparse,None,NaN,NaN,High (> 75%)
...,...,...,...,...,...,...,...,...,...,...,...
65,65,No,NaN,NaN,Normal/near-normal lung,Vessels (wall or lumen),None-sparse,None,NaN,NaN,High (> 75%)
66,66,No,NaN,NaN,Normal/near-normal lung,NaN,None-sparse,None,NaN,NaN,High (> 75%)
67,67,Yes,Acinar,Lepidic,NaN,NaN,Marked,None,Roughly equal,Moderate,High (> 75%)
68,68,Yes,Acinar,Micropapillary,NaN,NaN,Mild-moderate,None,Roughly equal,Low,High (> 75%)


In [167]:


# Load the tile-level data
tiles = pd.read_csv("/Users/vaidehipandya/Desktop/Work/csv_files/from_raw_data/TCGA_LUAD_5x_he_train_filtered_leiden_2p5__fold2.csv")

# Load the clean HPC-level consensus file
hpc_labels = pd.read_csv("HPC_consensus_labels_clean.csv")

print("Tiles shape:", tiles.shape)
print("HPC labels shape:", hpc_labels.shape)
print("\nTile columns sample:", tiles.columns[:10].tolist())
print("HPC label columns:", hpc_labels.columns.tolist())

Tiles shape: (360667, 4)
HPC labels shape: (71, 11)

Tile columns sample: ['samples', 'slides', 'tiles', 'leiden_2.5']
HPC label columns: ['hpc_id', 'malignant_label', 'malignant_subtype', 'second_malignant_subtype', 'non_malignant_subtype', 'second_non_malignant_subtype', 'inflammation_label', 'necrosis_label', 'stromal_epithelium_ratio', 'stromal_cellularity', 'cluster_homogeneity']


In [168]:
tiles.rename(columns={"leiden_2.5": "hpc_id"}, inplace=True)

In [169]:

# Save it back to disk
tiles.to_csv("TCGA_LUAD_5x_he_train_filtered_leiden_2p5__fold2.csv", index=False, encoding="utf-8")

print("Column renamed to 'hpc_id' and file saved successfully.")
print("Columns now:", tiles.columns[:10].tolist())

Column renamed to 'hpc_id' and file saved successfully.
Columns now: ['samples', 'slides', 'tiles', 'hpc_id']


In [171]:


df = pd.read_csv("TCGA_LUAD_5x_he_train_filtered_leiden_2p5__fold2.csv")
print("Columns:", df.columns.tolist()[:10])
print("Total rows:", len(df))

Columns: ['samples', 'slides', 'tiles', 'hpc_id']
Total rows: 360667


In [172]:
df["slide_tile"] = df["slides"].astype(str) + "_" + df["tiles"].astype(str)
print("Added slide_tile column.")

Added slide_tile column.


In [ ]:
dupe_count = df.duplicated(subset="slide_tile").sum()
print("Duplicate slide_tile entries:", dupe_count)
print("Unique slide_tile entries:", df["slide_tile"].nunique())

Duplicate slide_tile entries: 100
Unique slide_tile entries: 360567


In [174]:
df_clean = df.drop_duplicates(subset="slide_tile", keep="first")

In [178]:
df_clean.to_csv("TCGA_LUAD_5x_he_train_filtered_leiden_2p5__fold2.csv", index=False)

In [177]:
dupe_count = df_clean.duplicated(subset="slide_tile").sum()
print("Duplicate slide_tile entries:", dupe_count)
print("Unique slide_tile entries:", df["slide_tile"].nunique())

Duplicate slide_tile entries: 0
Unique slide_tile entries: 360567


In [179]:


# Load the tile-level dataset (the one you just renamed and saved)
tiles = pd.read_csv("TCGA_LUAD_5x_he_train_filtered_leiden_2p5__fold2.csv")

# Load the HPC-level consensus labels
consensus = pd.read_csv("HPC_consensus_labels_clean.csv")

print("Tiles shape:", tiles.shape)
print("HPC consensus shape:", consensus.shape)

Tiles shape: (360567, 5)
HPC consensus shape: (71, 11)


In [180]:
merged_tiles = tiles.merge(consensus, on="hpc_id", how="left")

print(" Merged successfully!")
print("Merged shape:", merged_tiles.shape)
print("Columns now:", merged_tiles.columns[:14].tolist())

 Merged successfully!
Merged shape: (360567, 15)
Columns now: ['samples', 'slides', 'tiles', 'hpc_id', 'slide_tile', 'malignant_label', 'malignant_subtype', 'second_malignant_subtype', 'non_malignant_subtype', 'second_non_malignant_subtype', 'inflammation_label', 'necrosis_label', 'stromal_epithelium_ratio', 'stromal_cellularity']


In [181]:
merged_tiles.to_csv("tile_level_training_data.csv", index=False, encoding="utf-8")
print("Saved merged tile-level dataset to tile_level_training_data.csv")

Saved merged tile-level dataset to tile_level_training_data.csv


In [182]:
print("Tiles with a matched label:", merged_tiles["malignant_label"].notna().sum())
print("Tiles without a match:", merged_tiles["malignant_label"].isna().sum())

Tiles with a matched label: 360567
Tiles without a match: 0


In [3]:

# Read the consensus file
results = pd.read_csv("tile_level_training_data.csv")

print(" File loaded successfully.")
print("Shape:", results.shape)
display(results.head(100))

 File loaded successfully.
Shape: (360567, 15)


,samples,slides,tiles,hpc_id,slide_tile,malignant_label,malignant_subtype,second_malignant_subtype,non_malignant_subtype,second_non_malignant_subtype,inflammation_label,necrosis_label,stromal_epithelium_ratio,stromal_cellularity,cluster_homogeneity
0,TCGA-55-7574,TCGA-55-7574-01Z-00-DX1,18_15.jpeg,32,TCGA-55-7574-01Z-00-DX1_18_15.jpeg,No,NaN,NaN,Collagenosis,NaN,Marked,None,NaN,NaN,High (> 75%)
1,TCGA-55-7574,TCGA-55-7574-01Z-00-DX1,14_6.jpeg,50,TCGA-55-7574-01Z-00-DX1_14_6.jpeg,Yes,Solid,Micropapillary,NaN,NaN,Mild-moderate,None,Roughly equal,Low,High (> 75%)
2,TCGA-55-7574,TCGA-55-7574-01Z-00-DX1,11_23.jpeg,27,TCGA-55-7574-01Z-00-DX1_11_23.jpeg,Yes,Acinar,Cribriform,Collagenosis,NaN,None-sparse,None,More stroma,Low,High (> 75%)
3,TCGA-55-7574,TCGA-55-7574-01Z-00-DX1,10_20.jpeg,21,TCGA-55-7574-01Z-00-DX1_10_20.jpeg,Yes,Solid,Acinar,NaN,NaN,Mild-moderate,None,Roughly equal,Moderate,High (> 75%)
4,TCGA-55-7574,TCGA-55-7574-01Z-00-DX1,23_1.jpeg,6,TCGA-55-7574-01Z-00-DX1_23_1.jpeg,No,NaN,NaN,Vessels (wall or lumen),NaN,None-sparse,None,NaN,NaN,High (> 75%)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,TCGA-55-7574,TCGA-55-7574-01Z-00-DX1,19_13.jpeg,3,TCGA-55-7574-01Z-00-DX1_19_13.jpeg,No,NaN,NaN,Collagenosis,Vessels (wall or lumen),None-sparse,None,NaN,NaN,High (> 75%)
96,TCGA-55-7574,TCGA-55-7574-01Z-00-DX1,11_13.jpeg,13,TCGA-55-7574-01Z-00-DX1_11_13.jpeg,Yes,Solid,Acinar,NaN,NaN,Mild-moderate,Some,More epithelium,Low,High (> 75%)
97,TCGA-55-7574,TCGA-55-7574-01Z-00-DX1,12_22.jpeg,27,TCGA-55-7574-01Z-00-DX1_12_22.jpeg,Yes,Acinar,Cribriform,Collagenosis,NaN,None-sparse,None,More stroma,Low,High (> 75%)
98,TCGA-55-7574,TCGA-55-7574-01Z-00-DX1,9_25.jpeg,42,TCGA-55-7574-01Z-00-DX1_9_25.jpeg,No,NaN,NaN,Collagenosis,Vessels (wall or lumen),None-sparse,None,NaN,NaN,High (> 75%)


In [184]:

# Load the tile-level data
df = pd.read_csv("tile_level_training_data.csv")

# Combine slide and tile to form a unique identifier
df["slide_tile"] = df["slides"].astype(str) + "_" + df["tiles"].astype(str)

print(" Added slide_tile column.")
print("Total rows:", len(df))
print("Unique slide_tile entries:", df['slide_tile'].nunique())
print("Duplicate slide_tile entries:", df.duplicated(subset='slide_tile').sum())

 Added slide_tile column.
Total rows: 360567
Unique slide_tile entries: 360567
Duplicate slide_tile entries: 0


In [186]:
dupes = df[df.duplicated(subset='slide_tile', keep=False)]
dupes_sorted = dupes.sort_values(["slides", "tiles"])
dupes_sorted.head(10)

,samples,slides,tiles,hpc_id,slide_tile,malignant_label,malignant_subtype,second_malignant_subtype,non_malignant_subtype,second_non_malignant_subtype,inflammation_label,necrosis_label,stromal_epithelium_ratio,stromal_cellularity,cluster_homogeneity


In [187]:
label_cols = [
    "malignant_label",
    "malignant_subtype",
    "second_malignant_subtype",
    "non_malignant_subtype",
    "second_non_malignant_subtype",
    "inflammation_label",
    "necrosis_label",
    "stromal_epithelium_ratio",
    "stromal_cellularity",
    "cluster_homogeneity"
]

conflicting = (
    dupes.groupby("slide_tile")[label_cols]
         .nunique()
         .max(axis=1)
)
conflict_tiles = conflicting[conflicting > 1].index
print("Tiles with conflicting labels:", len(conflict_tiles))

Tiles with conflicting labels: 0


In [4]:

#For Heatmap

# Drop exactly these columns
results = results.drop(columns=["slides", "tiles", "hpc_id", "samples"])

# Save the cleaned file
results.to_csv("tile_level_labels.csv", index=False)